# 01 -- Preprocessing

Data loading, cleaning, translation, dtype/memory fixes, and model-specific data preparation (dedup-before-split, tokenization) -- extracted directly from the real original notebook (`final code/Olist/olist_full_eda_preprocessing_PYTORCH.ipynb`), cells as indexed there, in original order, outputs preserved where the original cell had them.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
        alt = candidate / "Olist_Marketplace_Platform"
        if (alt / "backend" / "app").is_dir() and (alt / "data").is_dir():
            return alt
    raise RuntimeError("Could not locate the project root above this notebook.")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd


## 2. Data Loading

The Olist dataset is split across 9 CSV files. You have two options:

1. **Manual path (like the original notebook)** — set `MANUAL_BASE_PATH` below to the
   folder on your PC where you extracted the CSVs (e.g. after downloading from Kaggle and
   unzipping). This is exactly the same idea as the original
   `r"C:\Users\User1\Downloads\E-commerce\Dataset"` path, just written in a way that
   works on Windows, Mac, and Linux without escaping backslashes.
2. **Auto-detect** — if you leave `MANUAL_BASE_PATH = None`, the notebook searches a few
   common locations automatically (Kaggle input folder, a local `data/` folder, or the
   notebook's own folder).


In [2]:
# --- OPTION 1: set this to your own folder, exactly like the original notebook did ---
# Example (Windows):  MANUAL_BASE_PATH = r"C:\Users\User1\Downloads\E-commerce\Dataset"
# Example (Mac/Linux): MANUAL_BASE_PATH = "/Users/me/Downloads/E-commerce/Dataset"
MANUAL_BASE_PATH = None   # <-- put your path here between quotes, or leave None for auto-detect


In [ ]:
# FIX: this cell previously hard-coded an absolute Windows-only path here
# (base_path = r"C:\Users\...\Dataset"), completely ignoring MANUAL_BASE_PATH set
# above -- contradicting the "Option 1 (manual) / Option 2 (auto-detect)" the
# markdown above promises, and breaking on any machine but the original one.
# Real auto-detect: honors MANUAL_BASE_PATH if set, otherwise searches a few
# common locations relative to this notebook.
from pathlib import Path

_NOTEBOOK_DIR = Path.cwd()
_REQUIRED_FILE = "olist_customers_dataset.csv"
_CANDIDATE_DIRS = [
    MANUAL_BASE_PATH,
    _NOTEBOOK_DIR / "Dataset",
    _NOTEBOOK_DIR / "data" / "raw",
    _NOTEBOOK_DIR / "data",
    # This notebook's actual known location on this machine: a sibling
    # "E-commerce/Dataset" folder two levels up (Fake news/E-commerce/Dataset),
    # not inside this notebook's own folder.
    _NOTEBOOK_DIR.parent.parent / "E-commerce" / "Dataset",
    "/kaggle/input/brazilian-ecommerce",
]

base_path = None
for candidate in _CANDIDATE_DIRS:
    if candidate and (Path(candidate) / _REQUIRED_FILE).is_file():
        base_path = str(candidate)
        break

if base_path is None:
    raise FileNotFoundError(
        f"Could not find the Olist raw CSVs (looked for {_REQUIRED_FILE!r} in: "
        f"{[str(c) for c in _CANDIDATE_DIRS if c]}). Set MANUAL_BASE_PATH above to "
        "the folder containing the 9 Olist CSVs."
    )

print(f"Using dataset folder: {base_path}")

customers = pd.read_csv(f"{base_path}/olist_customers_dataset.csv")
geolocation = pd.read_csv(f"{base_path}/olist_geolocation_dataset.csv")
items = pd.read_csv(f"{base_path}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{base_path}/olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{base_path}/olist_order_reviews_dataset.csv")
orders_dataset = pd.read_csv(f"{base_path}/olist_orders_dataset.csv")
products = pd.read_csv(f"{base_path}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{base_path}/olist_sellers_dataset.csv")
translation = pd.read_csv(f"{base_path}/product_category_name_translation.csv")


In [4]:
orders_dataset.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [5]:
customers.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [6]:
items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87


In [7]:
products.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00


In [8]:
geolocation.head(3)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP


In [9]:
payments.head(3)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [10]:
sellers.head(3)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


In [11]:
translation.head(3)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [12]:
reviews.head(3)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24


---
## 3. Data Preprocessing

Cleaning is done **before** merging, table by table, so each fix is traceable and the
unified dataframe doesn't inherit ambiguous NaNs. The approach favors **business logic
over blind imputation**: a missing delivery date isn't a "dirty" value, it's a fact about
an order that was canceled and never shipped.

### 3.1 Duplicate Rows

In [13]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders_dataset,
    "products": products,
    "sellers": sellers,
    "translation": translation
}
for name, d in datasets.items():
    dup = d.duplicated().sum()

    if dup > 0:
        d.drop_duplicates(inplace=True)
        print(f"{name}: removed {dup} duplicate rows.")
    else:
        print(f"{name}: 0 duplicates.")

customers: 0 duplicates.
geolocation: removed 261831 duplicate rows.
items: 0 duplicates.
payments: 0 duplicates.
reviews: 0 duplicates.
orders: 0 duplicates.
products: 0 duplicates.
sellers: 0 duplicates.
translation: 0 duplicates.


### 3.2 Missing Values — Assessment

In [14]:
for name, d in datasets.items():
    nulls = d.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"--- {name} ---")
    print(nulls if not nulls.empty else "No missing values.")
    print()


--- customers ---
No missing values.

--- geolocation ---
No missing values.

--- items ---
No missing values.

--- payments ---
No missing values.

--- reviews ---
review_comment_title      88289
review_comment_message    58275
dtype: int64

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- sellers ---
No missing values.

--- translation ---
No missing values.



### 3.3 Missing Values — Treatment (with justification)

| Table | Column(s) | Decision | Why |
|---|---|---|---|
| Orders | `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date` | **Keep as NaT** | These orders were canceled/unavailable and genuinely never reached that stage. Imputing a fake date would corrupt delivery-time analysis. |
| Products | `product_category_name` | Fill with `'unknown'` | Category truly unknown, but row still represents a real sale — can't drop it. |
| Products | size/weight/photo columns | Fill with `0` | Keeps dtype numeric and consistent; these are physical specs missing only for a handful of products. |
| Reviews | `review_comment_title`, `review_comment_message` | Fill with `'No Title'` / `'No Message'` | Customer rated with stars but chose not to write text — that's a valid behavior, not missing data. |


In [15]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

num_product_cols = [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]
for col in num_product_cols:
    products[col] = products[col].fillna(0)

reviews["review_comment_title"] = reviews["review_comment_title"].fillna("No Title")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("No Message")

print("Remaining NaNs in Products:", products.isnull().sum().sum())
print("Remaining NaNs in Reviews :", reviews.isnull().sum().sum())


Remaining NaNs in Products: 0
Remaining NaNs in Reviews : 0


### 3.3b Translating Review Comments (Portuguese → English)

`review_comment_message` is written by customers in Portuguese, which is fine for
the word cloud in §4.6b but limits any English-only NLP tooling (e.g. feeding
comments into an English-trained classifier). This step adds an English column,
`review_comment_message_en`, translated via Google Translate (`deep-translator`).

**Notes:**
- Translation only runs on rows that actually have a written comment (skips the
  `"No Message"` placeholder from the previous cell).
- Progress is checkpointed to `reviews_translation_checkpoint.csv` every 50 rows,
  so an interrupted run (rate limit, closed kernel, etc.) can simply be re-run and
  will resume automatically instead of starting over.
- Translating all ~41k comments takes a few hours over the free API. Set
  `TRANSLATE_SAMPLE_SIZE` below to a smaller number (e.g. `2000`) for a quick run,
  or `None` to translate every commented review.

In [ ]:
# pip install transformers sentencepiece  (uncomment below if not already installed)
# %pip install -q transformers sentencepiece

import os
import time
import torch
from transformers import MarianMTModel, MarianTokenizer

# FIX: this cell previously called the online deep-translator (Google Translate /
# MyMemory) API per-row -- fragile (rate limits, needs live internet every run),
# not reproducible (a hosted translation service can change/degrade over time),
# and inconsistent with the model this project actually shipped: MarianMT
# ("Helsinki-NLP/opus-mt-ROMANCE-en"), a local, offline, deterministic
# HuggingFace model -- verified as the model that actually produced
# reviews_translated.csv (see ARTIFACT_AUDIT.md §4, backend/app/ml/translation.py).
TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-ROMANCE-en"
TRANSLATE_SAMPLE_SIZE = 2000   # set to None to translate ALL commented reviews (~41k, slower but still local/offline)
RANDOM_SEED = 42
BATCH_SIZE = 32
MAX_LENGTH = 512
SAVE_EVERY_BATCHES = 50

CHECKPOINT_CSV = os.path.join(base_path, "reviews_translation_checkpoint.csv") \
    if 'base_path' in globals() else "reviews_translation_checkpoint.csv"

to_translate = reviews[reviews["review_comment_message"] != "No Message"].copy()
print(f"Reviews with an actual written comment: {len(to_translate)}")

if TRANSLATE_SAMPLE_SIZE is not None and TRANSLATE_SAMPLE_SIZE < len(to_translate):
    to_translate = to_translate.sample(TRANSLATE_SAMPLE_SIZE, random_state=RANDOM_SEED)
    print(f"Sampled down to {len(to_translate)} rows (TRANSLATE_SAMPLE_SIZE={TRANSLATE_SAMPLE_SIZE})")

if os.path.exists(CHECKPOINT_CSV):
    print(f"Found checkpoint {CHECKPOINT_CSV}, resuming ...")
    done_df = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done_df["review_id"])
    remaining = to_translate[~to_translate["review_id"].isin(done_ids)]
    results = done_df.to_dict("records")
else:
    remaining = to_translate
    results = []

print(f"Remaining to translate: {len(remaining)}")

if len(remaining) > 0:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading translation model on {device}: {TRANSLATION_MODEL}")
    tokenizer = MarianTokenizer.from_pretrained(TRANSLATION_MODEL)
    # use_safetensors=True: recent transformers refuses torch.load on torch<2.6
    # (CVE-2025-32434) regardless of weights_only=True; this checkpoint publishes
    # a safetensors file that sidesteps the restriction entirely (same fix as the
    # BERT sentiment checkpoint above).
    model = MarianMTModel.from_pretrained(TRANSLATION_MODEL, use_safetensors=True).to(device)
    model.eval()

    remaining_records = remaining.to_dict("records")
    n_batches = (len(remaining_records) + BATCH_SIZE - 1) // BATCH_SIZE

    with torch.no_grad():
        for batch_idx in range(n_batches):
            start_i = batch_idx * BATCH_SIZE
            end_i = min(start_i + BATCH_SIZE, len(remaining_records))
            batch_rows = remaining_records[start_i:end_i]
            batch_texts = [str(r["review_comment_message"]) for r in batch_rows]
            safe_batch = [t if t.strip() else "." for t in batch_texts]

            inputs = tokenizer(
                safe_batch, return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_LENGTH,
            ).to(device)
            generated = model.generate(**inputs, max_length=MAX_LENGTH)
            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)

            for row, translated_text in zip(batch_rows, decoded):
                row["review_comment_message_en"] = translated_text
                results.append(row)

            if (batch_idx + 1) % SAVE_EVERY_BATCHES == 0 or (batch_idx + 1) == n_batches:
                pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False)
                print(f"  ... checkpointed at batch {batch_idx + 1}/{n_batches}")

    del model, tokenizer

translated_df = pd.DataFrame(results)[["review_id", "review_comment_message_en"]]
reviews = reviews.merge(translated_df, on="review_id", how="left")
reviews["review_comment_message_en"] = reviews["review_comment_message_en"].fillna("")

print(f"\nTranslated rows: {(reviews['review_comment_message_en'] != '').sum()} / {len(reviews)}")
reviews[["review_comment_message", "review_comment_message_en"]].dropna().head(8)


### 3.4 Data Type Corrections

In [17]:
orders_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in orders_date_cols:
    orders_dataset[col] = pd.to_datetime(orders_dataset[col], errors="coerce")

reviews_date_cols = ["review_creation_date", "review_answer_timestamp"]
for col in reviews_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

int_cols = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]
for col in int_cols:
    products[col] = products[col].astype("int64")

print(orders_dataset[orders_date_cols].dtypes)
print(reviews[reviews_date_cols].dtypes)
print(items["shipping_limit_date"].dtype)


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object
datetime64[us]


### 3.6 Geolocation Compression

The raw geolocation file has **1M+ rows** with heavy duplication (the same zip-code prefix
repeated with micro-differences in coordinates). We compress it to one row per zip prefix
using the mean lat/lng — a ~98% size reduction with no meaningful precision loss for
city/state-level analysis.

In [19]:
geo_cleaned = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": "first",
        "geolocation_state": "first",
    })
    .reset_index()
    .rename(columns={
        "geolocation_zip_code_prefix": "zip_code_prefix",
        "geolocation_lat": "lat",
        "geolocation_lng": "lng",
        "geolocation_city": "city",
        "geolocation_state": "state",
    })
)

print(f"Original rows : {len(geolocation):,}")
print(f"Compressed rows: {len(geo_cleaned):,}")
print(f"Reduction      : {(1 - len(geo_cleaned)/len(geolocation)):.1%}")
geo_cleaned.head(3)


Original rows : 738,332
Compressed rows: 19,015
Reduction      : 97.4%


,zip_code_prefix,lat,lng,city,state
0,1001,-23.55,-46.63,sao paulo,SP
1,1002,-23.55,-46.63,sao paulo,SP
2,1003,-23.55,-46.64,sao paulo,SP


### 3.7 Standardizing Brazilian State Codes → Full Names

In [20]:
brazil_states = {
    "SP": "São Paulo", "MG": "Minas Gerais", "ES": "Espírito Santo", "RS": "Rio Grande do Sul",
    "DF": "Distrito Federal", "PR": "Paraná", "SC": "Santa Catarina", "RJ": "Rio de Janeiro",
    "GO": "Goiás", "BA": "Bahia", "MA": "Maranhão", "PB": "Paraíba", "PE": "Pernambuco",
    "CE": "Ceará", "MT": "Mato Grosso", "PI": "Piauí", "RN": "Rio Grande do Norte",
    "MS": "Mato Grosso do Sul", "PA": "Pará", "AM": "Amazonas", "SE": "Sergipe",
    "RO": "Rondônia", "RR": "Roraima", "TO": "Tocantins", "AP": "Amapá",
    "AL": "Alagoas", "AC": "Acre",
}

sellers["seller_state"] = sellers["seller_state"].replace(brazil_states)
customers["customer_state"] = customers["customer_state"].replace(brazil_states)

print(customers["customer_state"].unique())


<ArrowStringArray>
[          'São Paulo',      'Santa Catarina',        'Minas Gerais',
              'Paraná',      'Rio de Janeiro',   'Rio Grande do Sul',
                'Pará',               'Goiás',      'Espírito Santo',
               'Bahia',            'Maranhão',  'Mato Grosso do Sul',
               'Ceará',    'Distrito Federal', 'Rio Grande do Norte',
          'Pernambuco',         'Mato Grosso',            'Amazonas',
               'Amapá',             'Alagoas',            'Rondônia',
             'Paraíba',           'Tocantins',               'Piauí',
                'Acre',             'Sergipe',             'Roraima']
Length: 27, dtype: str


### 3.8 Merging Into a Single Analysis-Ready DataFrame

Order-level grain. `payments` and `reviews` are aggregated to one row per order **before**
merging, otherwise the join would duplicate order rows and silently inflate revenue and
order counts.

In [21]:
payments_summary = payments.groupby("order_id", as_index=False)["payment_value"].sum()
payments_summary = payments_summary.rename(columns={"payment_value": "total_payment_value"})

payment_type_mode = (
    payments.groupby("order_id")["payment_type"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={"payment_type": "main_payment_type"})
)

installments_summary = payments.groupby("order_id", as_index=False)["payment_installments"].max()

reviews_summary = reviews.groupby("order_id", as_index=False)["review_score"].mean()

products_en = pd.merge(products, translation, on="product_category_name", how="left")

df = orders_dataset.merge(customers, on="customer_id", how="left")
df = df.merge(items, on="order_id", how="left")
df = df.merge(products_en, on="product_id", how="left")
df = df.merge(sellers, on="seller_id", how="left")
df = df.merge(payments_summary, on="order_id", how="left")
df = df.merge(payment_type_mode, on="order_id", how="left")
df = df.merge(installments_summary, on="order_id", how="left")
df = df.merge(reviews_summary, on="order_id", how="left")

print(f"Unified dataframe shape: {df.shape}")
df.head(3)


Unified dataframe shape: (113425, 34)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,total_payment_value,main_payment_type,payment_installments,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,São Paulo,1.00,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.00,268.00,4.00,500.00,19.00,8.00,13.00,housewares,"9,350.00",maua,São Paulo,38.71,voucher,1.00,4.00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,Bahia,1.00,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.00,178.00,1.00,400.00,19.00,13.00,19.00,perfumery,"31,570.00",belo horizonte,São Paulo,141.46,boleto,1.00,4.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,Goiás,1.00,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.00,232.00,1.00,420.00,24.00,19.00,21.00,auto,"14,840.00",guariba,São Paulo,179.12,credit_card,3.00,5.00


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 34 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   customer_unique_id             113425 non-null  str           
 9   customer_zip_code_prefix       113425 non-null  int32         
 10  customer_city                  113425 non-null  str           
 11  customer_st

### 3.9 Post-Merge Missing Values

After merging, a few new NaNs appear (e.g. rows with no matching product/payment).
Rule of thumb applied:
* **`product_id` missing** → the order-items join failed (rare); we drop those rows since
  there's nothing to analyze at the item level.
* **`product_category_name_english` missing** → keep the row, label category `'Unknown'`.
* **`review_score` missing** → no review was ever submitted; encode as `0` for an explicit
  "no review" bucket rather than dropping the order.

In [23]:
print("Missing values before fix:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df = df.dropna(subset=["product_id"])
df["product_category_name_english"] = df["product_category_name_english"].fillna("Unknown")
df["product_category_name"] = df["product_category_name"].fillna("Unknown")
df["review_score"] = df["review_score"].fillna(0)
df["total_payment_value"] = df["total_payment_value"].fillna(df["price"] + df["freight_value"])
df["main_payment_type"] = df["main_payment_type"].fillna("not_defined")
df["payment_installments"] = df["payment_installments"].fillna(1)

print("\nMissing values after fix:")
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "None (besides natural NaT delivery dates).")


Missing values before fix:
order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
product_category_name             775
product_name_lenght               775
product_description_lenght        775
product_photos_qty                775
product_weight_g                  775
product_length_cm                 775
product_height_cm                 775
product_width_cm                  775
product_category_name_english    2402
seller_zip_code_prefix            775
seller_city                       775
seller_state                      775
total_payment_value                 3
main_payment_type                   3
payment_installments                3
dtype: int64

Missing values after fix:
order_approved_at    

### 3.10 Feature Engineering

New columns that power the EDA below: delivery duration, delivery delay vs. estimate,
order hour/day/month, and an outlier flag for price using the IQR method.

In [24]:
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

# Delivery duration in days (NaT-safe: stays NaN for undelivered orders)
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days

# Positive = delivered late vs. estimate, negative = delivered early
df["delivery_delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
df["is_late_delivery"] = df["delivery_delay_days"] > 0

# Calendar features
df["order_hour"] = df["order_purchase_timestamp"].dt.hour
df["order_day"] = df["order_purchase_timestamp"].dt.day_name()
df["order_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

# Outlier flag on price using IQR (kept, not removed — flagged for transparency)
q1, q3 = df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["is_price_outlier"] = (df["price"] < lower) | (df["price"] > upper)

print(f"Price outliers flagged: {df['is_price_outlier'].sum():,} rows "
      f"({df['is_price_outlier'].mean():.1%} of data) — kept in dataset, just flagged.")

df[["delivery_days", "delivery_delay_days", "is_late_delivery", "order_hour", "order_day", "is_price_outlier"]].head()


Price outliers flagged: 8,427 rows (7.5% of data) — kept in dataset, just flagged.


,delivery_days,delivery_delay_days,is_late_delivery,order_hour,order_day,is_price_outlier
0,8.00,-8.00,False,10,Monday,False
1,13.00,-6.00,False,20,Tuesday,False
2,9.00,-18.00,False,8,Wednesday,False
3,13.00,-13.00,False,19,Saturday,False
4,2.00,-10.00,False,21,Tuesday,False


In [ ]:
# 6A. Data Preparation — Review Sentiment Split
from sklearn.model_selection import train_test_split
import re

# Build a clean text/label frame from the raw `reviews` table (order-review level).
# Exclude neutral (3-star) reviews and reviews with no written comment, so the model
# learns from genuinely polarized, non-empty text (same filter used for the §4.6b word clouds).
bert_df = reviews[
    (reviews["review_score"].isin([1, 2, 4, 5])) &
    (reviews["review_comment_message"] != "No Message")
].copy()

bert_df["label"] = bert_df["review_score"].apply(lambda s: 0 if s in [1, 2] else 1)
bert_df = bert_df[["review_comment_message", "label"]].rename(
    columns={"review_comment_message": "text"}
).reset_index(drop=True)

print(f"Usable labeled reviews (before dedup): {len(bert_df):,}")

# FIX -- real train/test leakage bug found in this exact cell: it used to call
# train_test_split() directly on `bert_df` here, and a later cell deduplicated into
# a NEW dataframe that these already-split X_train/X_val/X_test were never rebuilt
# from -- so every downstream BERT/CNN2D metric was computed on a test set
# containing rows the model had already seen in train. Verified on this dataset:
# 1,097 identical review texts were shared across the old train/val/test split.
#
# The fix: normalize text -> drop rows where the SAME normalized text carries both
# a Positive and a Negative label (unresolvable from text alone) -> drop duplicate
# normalized text -> THEN split. Never split before dedup again.
def _normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).strip().lower())

bert_df["normalized_text"] = bert_df["text"].map(_normalize_text)

label_counts = bert_df.groupby("normalized_text")["label"].nunique()
conflicting_texts = set(label_counts[label_counts > 1].index)
conflict_mask = bert_df["normalized_text"].isin(conflicting_texts)
print(f"Conflicting-label groups dropped: {len(conflicting_texts):,} groups, {int(conflict_mask.sum()):,} rows")
bert_df = bert_df.loc[~conflict_mask].reset_index(drop=True)

duplicate_groups = int((bert_df["normalized_text"].value_counts() > 1).sum())
bert_df = bert_df.drop_duplicates(subset=["normalized_text"], keep="first").reset_index(drop=True)
print(f"Duplicate-text groups collapsed: {duplicate_groups:,}")
print(f"Usable labeled reviews (after dedup): {len(bert_df):,}")
print(bert_df["label"].value_counts().rename({0: "Negative", 1: "Positive"}))

# 70 / 10 / 20 stratified split — same proportions used throughout this notebook,
# now run on the DEDUPLICATED frame.
X_temp, X_test, y_temp, y_test = train_test_split(
    bert_df["text"], bert_df["label"],
    test_size=0.20, random_state=42, stratify=bert_df["label"]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

print(f"\nTrain : {len(X_train):,}")
print(f"Val   : {len(X_val):,}")
print(f"Test  : {len(X_test):,}")

# Verification: zero raw-text overlap across splits (must be all zero).
train_txt, val_txt, test_txt = set(X_train), set(X_val), set(X_test)
overlap = {
    "train_val": len(train_txt & val_txt),
    "train_test": len(train_txt & test_txt),
    "val_test": len(val_txt & test_txt),
}
print("Text overlap across splits (must be all zero):", overlap)
assert all(v == 0 for v in overlap.values()), "Text leakage detected across splits!"


In [ ]:
# 6A.2 Tokenizer, Subsampling & PyTorch Dataset/DataLoader Setup
#!pip install -q "transformers==4.40.2"

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

print("====== [1] Initializing BERT Tokenizer ======")
# FIX: this cell previously used "distilbert-base-multilingual-cased" (DistilBERT).
# The checkpoint actually verified, fine-tuned, and shipped as this project's
# production BERT model is "LiYuan/amazon-review-sentiment-analysis" (a
# bert-base-multilingual-uncased architecture, NOT DistilBERT -- confirmed against
# the saved config.json's model_type). Training against the wrong base checkpoint
# here would produce a materially different model than the one this project's
# results/metrics actually describe.
BERT_CHECKPOINT = "LiYuan/amazon-review-sentiment-analysis"
tokenizer_bert = AutoTokenizer.from_pretrained(BERT_CHECKPOINT)

BERT_MAX_LEN = 128

# FIX: this cell previously subsampled to TRAIN_SIZE=5000 / VAL_SIZE=1000 /
# TEST_SIZE=2000 before training/evaluating BERT, while CNN2D (below) trains and
# evaluates on the FULL split -- making the "CNN2D vs. BERT" comparison in §6B.5 an
# apples-to-oranges comparison (BERT scored on 2,000 test rows, CNN2D on the full
# ~6,300 deduplicated rows). Both models now use the identical, full, deduplicated
# split from §6A -- see the fair-comparison-methodology note in this project's
# MODEL_COMPARISON_AUDIT.md.
X_train_bert_raw = X_train.astype(str).tolist()
X_val_bert_raw   = X_val.astype(str).tolist()
X_test_bert_raw  = X_test.astype(str).tolist()

y_train_bert = y_train.values
y_val_bert   = y_val.values
y_test_bert  = y_test.values


class ReviewSentimentDataset(Dataset):
    '''Holds raw text/label pairs; tokenization happens lazily in the collate function
    so each batch is padded only to its own longest sequence (dynamic padding).'''

    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])


def make_collate_fn(tokenizer, max_len=BERT_MAX_LEN):
    def collate_fn(batch):
        texts, labels = zip(*batch)
        encodings = tokenizer(
            list(texts),
            max_length=max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        encodings["labels"] = torch.tensor(labels, dtype=torch.long)
        return encodings
    return collate_fn


bert_collate_fn = make_collate_fn(tokenizer_bert, BERT_MAX_LEN)

train_dataset_bert = ReviewSentimentDataset(X_train_bert_raw, y_train_bert)
val_dataset_bert   = ReviewSentimentDataset(X_val_bert_raw, y_val_bert)
test_dataset_bert  = ReviewSentimentDataset(X_test_bert_raw, y_test_bert)

BERT_BATCH_SIZE = 8
train_loader_bert = DataLoader(train_dataset_bert, batch_size=BERT_BATCH_SIZE, shuffle=True, collate_fn=bert_collate_fn)
val_loader_bert   = DataLoader(val_dataset_bert, batch_size=BERT_BATCH_SIZE, shuffle=False, collate_fn=bert_collate_fn)
test_loader_bert  = DataLoader(test_dataset_bert, batch_size=BERT_BATCH_SIZE, shuffle=False, collate_fn=bert_collate_fn)

print("\n✔ Data preprocessing complete!")
print(f"-> Train: {len(train_dataset_bert):,} examples | {len(train_loader_bert)} batches")
print(f"-> Val:   {len(val_dataset_bert):,} examples | {len(val_loader_bert)} batches")
print(f"-> Test:  {len(test_dataset_bert):,} examples | {len(test_loader_bert)} batches")

In [55]:
# 6B.1 Tokenize & Pad Review Text (pure Python/NumPy — no Keras dependency)
import re
from collections import Counter

CNN_MAX_WORDS = 30_000
CNN_MAX_LEN = 100      # reviews are short; 100 tokens comfortably covers most comments
CNN_EMBEDDING_DIM = 100
OOV_TOKEN = "<OOV>"


def simple_tokenize(text: str):
    '''Lowercase word tokenizer — good enough for building a CNN vocabulary.'''
    return re.findall(r"\b\w+\b", str(text).lower())


class SimpleVocabTokenizer:
    '''Minimal stand-in for `keras.preprocessing.text.Tokenizer`: builds a frequency-capped
    word_index (0 reserved for padding, 1 for OOV) and converts texts to integer sequences.'''

    def __init__(self, num_words: int, oov_token: str = OOV_TOKEN):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {oov_token: 1}

    def fit_on_texts(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(simple_tokenize(text))
        for idx, (word, _) in enumerate(counter.most_common(self.num_words - 2), start=2):
            self.word_index[word] = idx

    def texts_to_sequences(self, texts):
        oov_idx = self.word_index[self.oov_token]
        return [[self.word_index.get(tok, oov_idx) for tok in simple_tokenize(text)] for text in texts]


def pad_sequences_np(sequences, maxlen: int, padding: str = "post", truncating: str = "post"):
    '''NumPy re-implementation of `keras.preprocessing.sequence.pad_sequences`.'''
    arr = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        if len(seq) > maxlen:
            seq = seq[:maxlen] if truncating == "post" else seq[-maxlen:]
        if padding == "post":
            arr[i, : len(seq)] = seq
        else:
            arr[i, -len(seq):] = seq
    return arr


cnn_tokenizer = SimpleVocabTokenizer(num_words=CNN_MAX_WORDS, oov_token=OOV_TOKEN)
cnn_tokenizer.fit_on_texts(X_train)


def cnn_encode(texts):
    seqs = cnn_tokenizer.texts_to_sequences(texts)
    return pad_sequences_np(seqs, maxlen=CNN_MAX_LEN, padding="post", truncating="post")


X_tr_seq = cnn_encode(X_train)
X_vl_seq = cnn_encode(X_val)
X_te_seq = cnn_encode(X_test)

print(f"Vocabulary size (capped at {CNN_MAX_WORDS}): {len(cnn_tokenizer.word_index):,}")
print(f"Train shape: {X_tr_seq.shape}  Val shape: {X_vl_seq.shape}  Test shape: {X_te_seq.shape}")

Vocabulary size (capped at 30000): 8,107
Train shape: (26642, 100)  Val shape: (3807, 100)  Test shape: (7613, 100)
